# L6: Hill-Climbing Loop

In Lesson 4, a grader sent each draft back for revision until it passed. That works, but it is expensive if the agent makes the *same* mistakes on every document. In this lesson you'll go one level up. Instead of fixing each output, you'll look at many runs, find the mistakes that keep recurring, and fix the **agent's instructions** so the mistakes stop happening.

This is called a **hill-climbing loop**, because you improve in small steps and keep a step only if the measurements go up:

1. **Run** the agent on a batch of documents and record the results
2. **Find** the problems that recur across runs
3. **Propose** one targeted change to the agent's instructions
4. **Apply** the change
5. **Re-run** the same batch and **compare**. Keep the change only if it helped.

**By the end of this lesson you will be able to:**
- Tell a one-off failure from a recurring pattern in a batch of runs
- Explain how an analyst model turns grader feedback into a proposed fix
- Read a diff to see exactly what changed in the agent's instructions
- Compare two versions of the agent using the same measurements
- Explain why one small, evidence-backed change at a time is a good strategy

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from run_history import generate_run_history
from hill_climbing import (
    propose_harness_change, load_proposal, apply_change,
    load_runs, load_summary, compare,
)
from visualization_helpers import (plot_hill_climbing_loop, 
    render_batch_overview, render_first_attempt_grid, 
    render_recurring_gaps, render_harness_change, render_comparison, 
    plot_before_after_pass_rates, render_report, render_requirement_doc,
    render_injected_feedback, render_diff,
)

%matplotlib inline
print("✓ Environment ready")

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"View"</em> option on the top menu of the notebook and then 2) click on <em>"File Browser"</em>.</p>

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em>.</p>
</div>

## The Run History

To spot patterns you need more than one example, so this lesson uses a batch of five requirement documents. Each one is vague in its own way, covering topics like authentication, notifications, payments, access control, and alert escalation.

The cell below shows the first document. The other four are commented out. Remove the `#` in front of a line if you want to read them too.

In [ ]:
display(render_requirement_doc("01_user_authentication"))
# display(render_requirement_doc("02_notification_system"))
# display(render_requirement_doc("03_payment_retry_policy"))
# display(render_requirement_doc("04_access_control_rules"))
# display(render_requirement_doc("05_alert_escalation_rules"))

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> LLMs do not always produce the same results. Consequently, the agent behavior you experience when running labs may differ from the video.</p>

##### Run the baseline (v1)

First we run the agent from Lesson 4, including its grader, on all five documents. This is our **baseline**, the starting point we'll try to beat. We call this version **v1**.

Two models are involved:
- `AGENT_MODEL`: does the work, writing the analysis for each document
- `ANALYST_MODEL`: used later, to study the results and suggest an improvement. Here it is a stronger model than the agent.

As in earlier lessons, `USE_LIVE_LLM` switches between running live and loading saved results. A live run takes longer than before, because it runs the agent on five documents, and later on all five again.

The table shows how many attempts each document needed. One attempt means the first draft passed. Two means the grader sent it back once.

In [ ]:
AGENT_MODEL = "openai:gpt-5.4-mini"
ANALYST_MODEL = "openai:gpt-5.5"
USE_LIVE_LLM = True

if USE_LIVE_LLM:
    generate_run_history(label="v1_live", model=AGENT_MODEL)
    label_v1 = "v1_live"
else:
    label_v1 = "v1"

runs_v1 = load_runs(label_v1)
summary_v1 = load_summary(label_v1)

display(render_batch_overview(runs_v1, summary_v1))

##### How did the first attempts do?

Every document eventually passed, so the final results look fine. The more telling question is: **how did each first draft do** before any revision? The grid shows one row per document and one column per rubric criterion (the same three criteria from Lesson 4).

Look at the bottom row, the pass rate for each criterion. If failures were random, they would be spread across the criteria and documents. If the *same* criteria fail again and again, the cause is probably in the agent's instructions, not bad luck.

In [ ]:
display(render_first_attempt_grid(runs_v1, summary_v1))

##### The feedback behind the failures

When a draft fails, the grader's feedback is sent back to the agent so it can revise. This is the same mechanism you saw in Lesson 4, and the next cell shows that feedback for one run.

Run 1 passed on its first attempt, so there is no feedback to show. To see feedback from a run that needed a revision, remove the `#` in front of one of the other lines.

In [ ]:
display(render_injected_feedback(runs_v1[0], 1, len(runs_v1)))
# display(render_injected_feedback(runs_v1[1], 2, len(runs_v1)))
# display(render_injected_feedback(runs_v1[2], 3, len(runs_v1)))
# display(render_injected_feedback(runs_v1[3], 4, len(runs_v1)))
# display(render_injected_feedback(runs_v1[4], 5, len(runs_v1)))

##### Recurring gaps

Now look at the feedback from *all* the runs at once. The next cell groups the grader's complaints by criterion and lists the gap found in each document.

Read down each group. Do the documents get the same complaint, in different words? A complaint that repeats across several documents is a **recurring gap**, and it is the best candidate for a fix. The pattern you see here is what the analyst will work from in the next section.

In [ ]:
display(render_recurring_gaps(runs_v1))

## Analyze the Traces

You could read all this feedback yourself, but here we hand it to the analyst model. It reads the run history, looks for recurring problems, and proposes a change to the agent's **harness**.

The *harness* is everything around the model that shapes its behavior. In this lesson that means the instruction file, `AGENTS.md`.

The analyst returns two things:
- `change`: a structured proposal, with specific fields we can apply automatically
- `proposal`: a written report explaining the reasoning

In [ ]:
if USE_LIVE_LLM:
    change, proposal = propose_harness_change(label_v1,model=ANALYST_MODEL)
else:
    change, proposal = load_proposal("v1")

##### The proposed change

The next cell prints each field of the structured proposal:

| Field | What it tells you |
|---|---|
| `target` | What is being changed (here, the agent's prompt) |
| `issue` | The problem the analyst found |
| `recurrence` | How many runs showed it |
| `evidence` | Quotes from the grader's feedback that back it up |
| `section` | Where in `AGENTS.md` the fix goes |
| `change` | The exact text to add |
| `rejected` | Problems the analyst noticed but chose *not* to fix |

A good proposal is specific and backed by evidence. Check that the `evidence` quotes really support the `issue`.

In [ ]:
from IPython.display import display, Markdown
for field, value in change.model_dump().items():
    display(Markdown(f"### {field}"))
    if isinstance(value, list):
        display(Markdown("\n".join(f"- {item}" for item in value)))
    else:
        display(Markdown(str(value)))

##### The analyst's written proposal

The same proposal, written as a report for a human reader. Three parts deserve attention:

- **The diagnosis.** It explains *why* the problem keeps happening. In the example run, the grader's rubric asked for lines that the agent's instructions never mentioned. The agent was being graded on something it was never told to do, so it only learned by being rejected.
- **The evidence.** Real feedback from real runs, not guesses.
- **The rejected observations.** The analyst noticed other issues but left them out because they weren't the main cause of the failures. Fixing one thing at a time keeps the effect of each change clear.

The analyst can be wrong, so a person should still review a proposal before it is adopted.

In [ ]:
display(render_report(proposal, title="proposal.md"))

## Improve the Harness

Now we apply the proposal. `apply_change` takes the current instructions (v1) and inserts the analyst's text, producing a new version (v2).

The cell shows the result as a **diff**, a view that highlights exactly what differs between two files. Lines starting with `+` were added, and lines starting with `-` were removed.

In [ ]:
baseline = Path("workspace/memories/AGENTS.md").read_text()
improved = apply_change(change, baseline)

display(render_diff(baseline, improved, 
                    from_label="AGENTS.md (v1)",to_label="AGENTS.md (v2)"))

Notice how small the change is. Only a few lines were added, in one section, and the rest of the file is untouched. That is deliberate. A small change is easy to review, and if the results improve, you know what caused it.

## Did It Work?

Hill-climbing only counts if the change actually helps, so we test it. We run the agent on the **same five documents** with the new v2 instructions, then compare v1 and v2 using the same measurements.

The comparison table shows:
- **Verdict:** whether the change helped overall
- **Pass rates per criterion:** how often first drafts passed, before and after
- **Revisions Needed:** how many documents had to be sent back
- **Attempts, tokens, and cost:** the price of getting each result

The grid below it shows the v2 first attempts, in the same format as before. Compare it with the v1 grid from earlier.

Exact numbers will vary from run to run.

In [ ]:
if USE_LIVE_LLM:
    generate_run_history(label="v2_live", model=AGENT_MODEL,
                         agents_md=Path("run_history/v2/AGENTS.md"))
    label_v2 = "v2_live"
else:
    label_v2 = "v2"

result = compare(label_v1, label_v2)
display(render_comparison(result, label_v1, label_v2))

runs_v2 = load_runs(label_v2)
summary_v2 = load_summary(label_v2)
display(render_first_attempt_grid(runs_v2, summary_v2))

##### Before and after

The same comparison as a chart, showing the pass rates for v1 and v2 side by side.

Notice that one fix to the instructions helped the criteria that kept failing. It also lowered the work and cost per document: fewer revisions means fewer attempts, and fewer attempts mean fewer tokens.

In [ ]:
fig = plot_before_after_pass_rates(result)
plt.show()

## Conclusion

In this lesson you stepped back from individual outputs and improved the agent itself.

**Key takeaways**
- **Fix the cause, not each instance.** In Lesson 4 the grader fixed every document one at a time. Here you asked why the same fixes were needed again and again, and fixed the instructions.
- **Look across runs.** A failure that repeats is a pattern. A failure that happens once might be noise. Recurrence is what tells you where to act.
- **Back proposals with evidence.** The analyst's change was tied to real grader feedback, which made it easy to check.
- **Change one thing at a time.** A small, targeted change is easy to review, and you can tell what caused any improvement.
- **Always measure again.** You kept the change because the same batch, scored the same way, came out better. That comparison is what makes this hill climbing.
- **Better instructions save work.** Fewer revisions meant fewer attempts and lower cost.

**Three loops, three jobs**
- The **agent loop** (Lesson 3) gets one task done
- The **verification loop** (Lesson 4) checks each result and sends it back until it passes
- The **hill-climbing loop** (Lesson 6) improves the agent so that less checking and revising is needed in the first place

**Keep in mind**
- Five documents is a small sample. Improvements may not carry over to different kinds of documents, so test on more before trusting a change.
- Hill-climbing improves what you measure. If the rubric is incomplete, the agent gets better at the rubric, not necessarily at the real task.
- Review proposed changes yourself before adopting them.

**Try it yourself**
- Uncomment other documents or feedback cells to see how the failures differ between runs
- Run the loop again with the v2 instructions as the new baseline and see whether the analyst finds anything else worth fixing